<a href="https://colab.research.google.com/github/leon-asim/EEG-Attention/blob/main/inssiok_eeg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import os
import copy
import random
import numpy as np
import pandas as pd
import scipy.io

from collections import Counter
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torch.nn.functional as F

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("inancigdem/eeg-data-for-mental-attention-state-detection")

path = os.path.join(path, 'EEG Data')

print("Path to dataset files:", path)

100%|██████████| 557M/557M [00:06<00:00, 86.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/inancigdem/eeg-data-for-mental-attention-state-detection/versions/1/EEG Data


In [5]:
!pip install scipy

In [6]:
data_root = path
files = os.listdir(data_root)
len(files)

34

In [7]:
FOCUSED_CLASS = 0
UNFOCUSED_CLASS = 1
DROWSY_CLASS = 2

HZ = 128
MAX_MINUTES = 30
MAX_SAMPLES = MAX_MINUTES * 60 * HZ

EEG_COLUMNS = [
    'AF3', 'F7', 'F3', 'FC5',
    'T7', 'P7', 'O1', 'O2',
    'P8', 'T8', 'FC6', 'F4',
    'F8', 'AF4'
]


def get_state(timestamp):
    """
    0-10 min  -> Focused
    10-20 min -> Unfocused
    20-30 min -> Drowsy
    """

    if timestamp < 10 * 60 * HZ:
        return FOCUSED_CLASS

    elif timestamp < 20 * 60 * HZ:
        return UNFOCUSED_CLASS

    elif timestamp < 30 * 60 * HZ:
        return DROWSY_CLASS

    else:
        return None

In [8]:
columns = [
    'ED_COUNTER',
    'ED_INTERPOLATED',
    'ED_RAW_CQ',
    'ED_AF3',
    'ED_F7',
    'ED_F3',
    'ED_FC5',
    'ED_T7',
    'ED_P7',
    'ED_O1',
    'ED_O2',
    'ED_P8',
    'ED_T8',
    'ED_FC6',
    'ED_F4',
    'ED_F8',
    'ED_AF4',
    'ED_GYROX',
    'ED_GYROY',
    'ED_TIMESTAMP',
    'ED_ES_TIMESTAMP',
    'ED_FUNC_ID',
    'ED_FUNC_VALUE',
    'ED_MARKER',
    'ED_SYNC_SIGNAL'
]


def get_EEG_data(data_root, filename):

    mat = scipy.io.loadmat(
        os.path.join(data_root, filename)
    )

    data = mat["o"]["data"][0, 0]

    eeg_df = pd.DataFrame(
        data,
        columns=columns
    )

    # Keep only EEG channels
    eeg_df = eeg_df[
        [
            'ED_AF3', 'ED_F7', 'ED_F3', 'ED_FC5',
            'ED_T7', 'ED_P7', 'ED_O1', 'ED_O2',
            'ED_P8', 'ED_T8', 'ED_FC6', 'ED_F4',
            'ED_F8', 'ED_AF4'
        ]
    ]

    eeg_df.columns = EEG_COLUMNS

    # Keep only first 30 minutes
    eeg_df = eeg_df.iloc[:MAX_SAMPLES].copy()

    # Standardize each recording
    scaler = StandardScaler()

    eeg_df[EEG_COLUMNS] = scaler.fit_transform(
        eeg_df[EEG_COLUMNS]
    )

    # Timestamp = sample number
    eeg_df.reset_index(inplace=True)
    eeg_df.rename(
        columns={'index': 'timestamp'},
        inplace=True
    )

    # Assign state
    eeg_df['state'] = eeg_df['timestamp'].apply(get_state)

    # Remove anything after 30 minutes
    eeg_df = eeg_df.dropna(subset=['state'])

    eeg_df['state'] = eeg_df['state'].astype(int)

    return eeg_df

In [9]:
dataset = []
# For each file, print # minutes of data
for filename in files:
    data = get_EEG_data(data_root, filename)
    dataset.append(data)

In [10]:
dataset[0]

,timestamp,AF3,F7,F3,FC5,T7,P7,O1,O2,P8,T8,FC6,F4,F8,AF4,state
0,0,-1.056992,1.667356,-0.487519,-0.171303,-0.071104,-0.219934,2.234175,0.152238,-0.527152,-0.402256,0.163455,2.642163,-1.108954,-1.195540,0
1,1,-1.056992,1.604205,-0.499922,-1.099214,-0.071104,-0.157113,2.220759,0.137697,-0.615588,-0.402256,0.163455,2.642163,-0.085403,-1.161823,0
2,2,-0.366131,1.730508,-0.537134,0.385444,-0.071104,-0.164966,2.287842,0.174050,-0.767192,-0.402256,0.183083,0.625213,-1.108954,-1.251736,0
3,3,-0.366131,1.477901,-0.580548,1.127774,-0.071104,-0.227787,2.234175,0.086802,-0.792460,-0.402256,0.124198,3.852333,-3.156058,-1.375367,0
4,4,-1.056992,1.288446,-0.530932,-0.171303,-0.071104,-0.227787,2.100009,-0.044069,-0.741925,-0.402256,0.104570,4.659113,-2.132506,-1.296693,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230395,230395,2.627598,-0.227197,0.349748,-1.655961,-0.998603,-0.031472,-0.167399,-0.298542,-0.325013,0.658908,0.448066,-0.584958,-0.085403,0.029533,2
230396,230396,2.857885,-0.227197,0.331142,-2.398291,-0.998603,-0.078587,-0.301565,-0.414872,-0.400815,1.295606,0.408809,1.431993,-0.085403,-0.004184,2
230397,230397,0.785303,-0.037742,0.312536,-2.212708,-0.998603,-0.125703,-0.395481,-0.480308,-0.413448,1.932304,0.389181,1.835383,-0.597179,-0.026663,2
230398,230398,-0.596418,-0.037742,0.312536,0.014280,-0.998603,-0.078587,-0.328398,-0.400331,-0.375547,1.083373,0.418623,0.625213,-0.085403,0.018294,2


In [11]:
print((dataset[0].state == 2).sum())

76800


In [12]:
type(dataset), type(dataset[0])

(list, pandas.core.frame.DataFrame)

In [13]:
print(len(files))
print(files)

34
['eeg_record6.mat', 'eeg_record16.mat', 'eeg_record10.mat', 'eeg_record7.mat', 'eeg_record2.mat', 'eeg_record34.mat', 'eeg_record1.mat', 'eeg_record22.mat', 'eeg_record11.mat', 'eeg_record29.mat', 'eeg_record15.mat', 'eeg_record33.mat', 'eeg_record18.mat', 'eeg_record30.mat', 'eeg_record8.mat', 'eeg_record19.mat', 'eeg_record12.mat', 'eeg_record5.mat', 'eeg_record14.mat', 'eeg_record4.mat', 'eeg_record17.mat', 'eeg_record3.mat', 'eeg_record31.mat', 'eeg_record13.mat', 'eeg_record20.mat', 'eeg_record9.mat', 'eeg_record28.mat', 'eeg_record26.mat', 'eeg_record27.mat', 'eeg_record21.mat', 'eeg_record32.mat', 'eeg_record25.mat', 'eeg_record23.mat', 'eeg_record24.mat']


34 different experiments and we are going to split them 70/15/15 for training, validating and testing.

In [15]:
# Fixed order
files = sorted(files)

# 70% train, 30% temporary
train_files, temp_files = train_test_split(
    files,
    test_size=0.30,
    random_state=42
)

# Split remaining 30% into 15% validation + 15% test
val_files, test_files = train_test_split(
    temp_files,
    test_size=0.50,
    random_state=42
)

print("Train:", len(train_files))
print("Validation:", len(val_files))
print("Test:", len(test_files))

print("\nTrain subjects:", train_files)
print("\nValidation subjects:", val_files)
print("\nTest subjects:", test_files)

Train: 23
Validation: 5
Test: 6

Train subjects: ['eeg_record13.mat', 'eeg_record24.mat', 'eeg_record25.mat', 'eeg_record14.mat', 'eeg_record21.mat', 'eeg_record2.mat', 'eeg_record10.mat', 'eeg_record11.mat', 'eeg_record6.mat', 'eeg_record12.mat', 'eeg_record5.mat', 'eeg_record30.mat', 'eeg_record7.mat', 'eeg_record3.mat', 'eeg_record26.mat', 'eeg_record32.mat', 'eeg_record15.mat', 'eeg_record28.mat', 'eeg_record9.mat', 'eeg_record16.mat', 'eeg_record19.mat', 'eeg_record22.mat', 'eeg_record4.mat']

Validation subjects: ['eeg_record8.mat', 'eeg_record17.mat', 'eeg_record20.mat', 'eeg_record33.mat', 'eeg_record29.mat']

Test subjects: ['eeg_record31.mat', 'eeg_record23.mat', 'eeg_record18.mat', 'eeg_record1.mat', 'eeg_record34.mat', 'eeg_record27.mat']


In [16]:
def split_epochs(data, hz=128, epoch_length=2, step_size=1):
    epoch_samples = int(epoch_length * hz)  # 256
    step_samples = int(step_size * hz)     # 128

    epochs = []

    for start in range(0, len(data) - epoch_samples + 1, step_samples):
        epoch = data.iloc[start:start + epoch_samples].copy()

        # Don't allow an epoch to contain multiple states
        if epoch['state'].nunique() == 1:
            epochs.append(epoch)

    return epochs

In [17]:
MAX_EPOCHS_PER_SUBJECT = 1000

def create_epochs(file_list):
    all_epochs = []

    for filename in file_list:
        print("Processing:", filename)

        data = get_EEG_data(data_root, filename)
        epochs = split_epochs(data)

        # Randomly select at most 1000 epochs
        if len(epochs) > MAX_EPOCHS_PER_SUBJECT:
            rng = np.random.default_rng(42)
            indices = rng.choice(
                len(epochs),
                MAX_EPOCHS_PER_SUBJECT,
                replace=False
            )
            epochs = [epochs[i] for i in indices]

        all_epochs.extend(epochs)

    return all_epochs

In [18]:
train_epochs = create_epochs(train_files)
val_epochs = create_epochs(val_files)
test_epochs = create_epochs(test_files)

print("Train epochs:", len(train_epochs))
print("Validation epochs:", len(val_epochs))
print("Test epochs:", len(test_epochs))

Processing: eeg_record13.mat
Processing: eeg_record24.mat
Processing: eeg_record25.mat
Processing: eeg_record14.mat
Processing: eeg_record21.mat
Processing: eeg_record2.mat
Processing: eeg_record10.mat
Processing: eeg_record11.mat
Processing: eeg_record6.mat
Processing: eeg_record12.mat
Processing: eeg_record5.mat
Processing: eeg_record30.mat
Processing: eeg_record7.mat
Processing: eeg_record3.mat
Processing: eeg_record26.mat
Processing: eeg_record32.mat
Processing: eeg_record15.mat
Processing: eeg_record28.mat
Processing: eeg_record9.mat
Processing: eeg_record16.mat
Processing: eeg_record19.mat
Processing: eeg_record22.mat
Processing: eeg_record4.mat
Processing: eeg_record8.mat
Processing: eeg_record17.mat
Processing: eeg_record20.mat
Processing: eeg_record33.mat
Processing: eeg_record29.mat
Processing: eeg_record31.mat
Processing: eeg_record23.mat
Processing: eeg_record18.mat
Processing: eeg_record1.mat
Processing: eeg_record34.mat
Processing: eeg_record27.mat
Train epochs: 23000
Val

In [19]:
class EEGDataset(Dataset):

    def __init__(self, dataframes):

        self.data = []
        self.targets = []

        for df in dataframes:

            target = int(
                df['state'].iloc[0]
            )

            features = df.drop(
                columns=['state', 'timestamp'],
                errors='ignore'
            )

            self.data.append(
                features.values
            )

            self.targets.append(target)

        self.data = torch.tensor(
            np.array(self.data),
            dtype=torch.float32
        )

        self.targets = torch.tensor(
            self.targets,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return (
            self.data[idx],
            self.targets[idx]
        )

In [20]:
train_dataset = EEGDataset(train_epochs)
val_dataset = EEGDataset(val_epochs)
test_dataset = EEGDataset(test_epochs)

In [21]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [22]:
print(train_dataset.data.shape)
print(train_dataset.targets.shape)

print(val_dataset.data.shape)
print(test_dataset.data.shape)

torch.Size([23000, 256, 14])
torch.Size([23000])
torch.Size([5000, 256, 14])
torch.Size([6000, 256, 14])


In [23]:
CHANNEL_CONFIGS = {

    14: [
        'AF3', 'F7', 'F3', 'FC5',
        'T7', 'P7', 'O1', 'O2',
        'P8', 'T8', 'FC6', 'F4',
        'F8', 'AF4'
    ],

    8: [
        'AF3', 'F7', 'F3', 'FC5',
        'FC6', 'F4', 'F8', 'AF4'
    ],

    4: [
        'AF3', 'F3',
        'F4', 'AF4'
    ],

    2: [
        'AF3', 'AF4'
    ],

    1: [
        'AF3'
    ]
}

In [24]:
def select_channels(epochs, channels):

    return [
        epoch[
            ['timestamp'] +
            channels +
            ['state']
        ]
        for epoch in epochs
    ]

In [34]:
class BiLSTM(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size=256,
        num_layers=2,
        num_classes=3
    ):

        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )

        self.fc1 = nn.Linear(
            hidden_size * 2,
            hidden_size
        )

        self.fc2 = nn.Linear(
            hidden_size,
            hidden_size // 2
        )

        self.fc3 = nn.Linear(
            hidden_size // 2,
            num_classes
        )

        self.dropout = nn.Dropout(0.5)

    def forward(self, x):

        _, (h_n, _) = self.lstm(x)

        # Last forward hidden state
        forward_hidden = h_n[-2]

        # Last backward hidden state
        backward_hidden = h_n[-1]

        # Explicitly combine BOTH directions
        out = torch.cat(
            [forward_hidden, backward_hidden],
            dim=1
        )

        out = F.relu(
            self.fc1(out)
        )

        out = F.relu(
            self.fc2(out)
        )

        out = self.dropout(out)

        out = self.fc3(out)

        return out

In [35]:
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    num_epochs=10
):

    best_val_loss = float("inf")
    best_model_state = None

    train_losses = []
    val_losses = []
    train_f1s = []
    val_f1s = []

    for epoch in range(num_epochs):

        # =========================
        # TRAIN
        # =========================

        model.train()

        running_loss = 0.0

        train_preds = []
        train_labels = []

        for data, targets in tqdm(
            train_loader,
            leave=False
        ):

            data = data.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()

            outputs = model(data)

            loss = criterion(
                outputs,
                targets
            )

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

            preds = outputs.argmax(
                dim=1
            )

            train_preds.extend(
                preds.cpu().numpy()
            )

            train_labels.extend(
                targets.cpu().numpy()
            )

        train_loss = (
            running_loss /
            len(train_loader)
        )

        train_f1 = f1_score(
            train_labels,
            train_preds,
            average="macro"
        )

        # =========================
        # VALIDATION
        # =========================

        model.eval()

        running_loss = 0.0

        val_preds = []
        val_labels = []

        with torch.no_grad():

            for data, targets in val_loader:

                data = data.to(device)
                targets = targets.to(device)

                outputs = model(data)

                loss = criterion(
                    outputs,
                    targets
                )

                running_loss += loss.item()

                preds = outputs.argmax(
                    dim=1
                )

                val_preds.extend(
                    preds.cpu().numpy()
                )

                val_labels.extend(
                    targets.cpu().numpy()
                )

        val_loss = (
            running_loss /
            len(val_loader)
        )

        val_f1 = f1_score(
            val_labels,
            val_preds,
            average="macro"
        )

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_f1s.append(train_f1)
        val_f1s.append(val_f1)

        print(
            f"Epoch {epoch+1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train F1: {train_f1:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val F1: {val_f1:.4f}"
        )

        # =========================
        # SAVE BEST MODEL
        # =========================

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_model_state = copy.deepcopy(
                model.state_dict()
            )

    # VERY IMPORTANT:
    # return the best model, not the final epoch
    model.load_state_dict(
        best_model_state
    )

    return model

In [36]:
def evaluate_model(model, test_loader):

    model.eval()

    test_preds = []
    test_labels = []

    with torch.no_grad():

        for data, targets in test_loader:

            data = data.to(device)

            outputs = model(data)

            preds = outputs.argmax(
                dim=1
            )

            test_preds.extend(
                preds.cpu().numpy()
            )

            test_labels.extend(
                targets.numpy()
            )

    report = classification_report(
        test_labels,
        test_preds,
        target_names=[
            "Focused",
            "Unfocused",
            "Drowsy"
        ],
        output_dict=True,
        zero_division=0
    )

    matrix = confusion_matrix(
        test_labels,
        test_preds
    )

    print(
        classification_report(
            test_labels,
            test_preds,
            target_names=[
                "Focused",
                "Unfocused",
                "Drowsy"
            ],
            zero_division=0
        )
    )

    print("Confusion matrix:")
    print(matrix)

    return report, matrix

In [37]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cuda


In [39]:
RESULTS_FILE = "bilstm_channel_results.csv"

all_results = []

for num_channels, channels in CHANNEL_CONFIGS.items():

    print("\n" + "=" * 60)
    print(f"RUNNING {num_channels} CHANNELS")
    print(channels)
    print("=" * 60)

    # -------------------------
    # Select channels
    # -------------------------

    train_selected = select_channels(
        train_epochs,
        channels
    )

    val_selected = select_channels(
        val_epochs,
        channels
    )

    test_selected = select_channels(
        test_epochs,
        channels
    )

    # -------------------------
    # Create datasets
    # -------------------------

    train_dataset = EEGDataset(
        train_selected
    )

    val_dataset = EEGDataset(
        val_selected
    )

    test_dataset = EEGDataset(
        test_selected
    )

    # -------------------------
    # DataLoaders
    # -------------------------

    generator = torch.Generator()

    generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True,
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=32,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False
    )

    # -------------------------
    # Class weights
    # -------------------------

    classes = np.array([0, 1, 2])

    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=train_dataset.targets.numpy()
    )

    class_weights = torch.tensor(
        weights,
        dtype=torch.float32
    ).to(device)

    # -------------------------
    # Model
    # -------------------------

    model = BiLSTM(
        input_size=num_channels,
        hidden_size=256,
        num_layers=2,
        num_classes=3
    ).to(device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    # -------------------------
    # Train
    # -------------------------

    model = train_model(
        model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        num_epochs=10
    )

    # -------------------------
    # Test BEST model
    # -------------------------

    report, matrix = evaluate_model(
        model,
        test_loader
    )

    # -------------------------
    # Save results
    # -------------------------

    result = {
        "channels": num_channels,
        "channel_names": ",".join(channels),

        "accuracy": report["accuracy"],

        "macro_precision": report[
            "macro avg"
        ]["precision"],

        "macro_recall": report[
            "macro avg"
        ]["recall"],

        "macro_f1": report[
            "macro avg"
        ]["f1-score"],

        "focused_f1": report[
            "Focused"
        ]["f1-score"],

        "unfocused_f1": report[
            "Unfocused"
        ]["f1-score"],

        "drowsy_f1": report[
            "Drowsy"
        ]["f1-score"]
    }

    all_results.append(result)

    # Save after EVERY experiment
    pd.DataFrame(all_results).to_csv(
        RESULTS_FILE,
        index=False
    )

    print("\nResults saved to:", RESULTS_FILE)


RUNNING 14 CHANNELS
['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']


Epoch 1/10 | Train Loss: 0.8497 | Train F1: 0.6053 | Val Loss: 0.8592 | Val F1: 0.5744


Epoch 2/10 | Train Loss: 0.8042 | Train F1: 0.6324 | Val Loss: 0.8745 | Val F1: 0.5777


Epoch 3/10 | Train Loss: 0.7357 | Train F1: 0.6738 | Val Loss: 0.8788 | Val F1: 0.5404


Epoch 4/10 | Train Loss: 0.7060 | Train F1: 0.6853 | Val Loss: 0.8355 | Val F1: 0.6046


...

Epoch 6/10 | Train Loss: 0.9970 | Train F1: 0.4796 | Val Loss: 1.0124 | Val F1: 0.4519


Epoch 7/10 | Train Loss: 0.9899 | Train F1: 0.4856 | Val Loss: 0.9516 | Val F1: 0.5106


Epoch 8/10 | Train Loss: 0.9521 | Train F1: 0.5171 | Val Loss: 0.8948 | Val F1: 0.5212


Epoch 9/10 | Train Loss: 0.9185 | Train F1: 0.5370 | Val Loss: 0.8909 | Val F1: 0.5672


Epoch 10/10 | Train Loss: 0.9031 | Train F1: 0.5505 | Val Loss: 0.9346 | Val F1: 0.4896
              precision    recall  f1-score   support

     Focused       0.71      0.49      0.58      2052
   Unfocused       0.41      0.68      0.51      2028
      Drowsy       0.55      0.35      0.43      1920

    accuracy                           0.51      6000
   macro avg       0.56      0.51      0.51      6000
weighted avg       0.56      0.51      0.51      6000

Confusion matrix:
[[1007  897  148]
 [ 248 1379  401]
 [ 166 1076  678]]

Results saved to: bilstm_channel_results.csv
